# Historical Options Greeks Backtest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apextick/apextick-starter-kit/blob/main/notebooks/02_historical_options_greeks_backtest.ipynb)

Query a **live option chain** with implied volatility and analytical Black-Scholes Greeks (delta, gamma, theta, vega), pick a contract, then pull its **daily OHLCV + IV + delta history** to study how a simple short-call position behaved.

**Plan & quota:** option chains and contract history need Trader Pro or above (contract history covers the trailing 90 days; longer and expired-contract history needs Quant & Community or above). The notebook uses about 4 requests.

In [ ]:
# The official client uses only the standard library; pandas and matplotlib come with Colab.
%pip install -q "git+https://github.com/apextick/apextick-starter-kit.git#subdirectory=python"

## Add your API key

In Colab, open **Secrets** (the key icon in the left sidebar), add `APEXTICK_API_KEY` with your key and allow this notebook to use it. Anywhere else, set the `APEXTICK_API_KEY` environment variable, or paste the key when prompted. No key yet? Get a free one at [dash.apextick.com/register](https://dash.apextick.com/register).

In [ ]:
import os
from getpass import getpass

try:  # Colab Secrets
    from google.colab import userdata
    os.environ.setdefault("APEXTICK_API_KEY", userdata.get("APEXTICK_API_KEY") or "")
except Exception:
    pass
if not os.environ.get("APEXTICK_API_KEY"):
    os.environ["APEXTICK_API_KEY"] = getpass("ApexTick API key (dk_...): ")

import pandas as pd
import matplotlib.pyplot as plt
from apextick import ApexTickAPIError, ApexTickClient, to_dataframe

client = ApexTickClient()  # reads APEXTICK_API_KEY
client

## 1. Live option chain with Greeks

`client.options_chain()` returns every listed contract (or one side, with `contract_type="call"` or `"put"`) with bid/ask, volume, open interest, implied volatility and Greeks. `to_dataframe()` flattens the nested Greeks into columns.

In [ ]:
UNDERLYING = "SPY"

try:
    spot = client.quote(UNDERLYING)[0]["price"]
    chain = client.options_chain(UNDERLYING, contract_type="call", limit=None)
except ApexTickAPIError as err:
    if err.status_code == 403:
        raise RuntimeError(f"Option chains need Trader Pro or above ({err.message}). Upgrade at https://dash.apextick.com/billing/pricing") from None
    raise

calls = to_dataframe(chain, "options").rename(columns={
    "implied_volatility": "iv", "greeks.delta": "delta", "greeks.gamma": "gamma",
    "greeks.theta": "theta", "greeks.vega": "vega",
})
# OCC symbols end in YYMMDD + C/P + strike x 1000, so the expiration is always recoverable.
calls["expiration"] = pd.to_datetime(calls["occ_symbol"].str.strip().str[-15:-9], format="%y%m%d")
calls["moneyness_pct"] = (calls["strike"] / spot - 1) * 100

print(f"{UNDERLYING} at {spot:.2f}: {len(calls)} call contracts across {calls['expiration'].nunique()} expirations")
calls[["occ_symbol", "expiration", "strike", "bid", "ask", "iv", "delta", "gamma", "theta", "vega", "open_interest"]].sort_values(["expiration", "strike"]).head(15)

In [ ]:
# Volatility smile and delta curve for the nearest expiration
nearest = calls[calls["expiration"] == calls["expiration"].min()].sort_values("strike")
fig, (ax_iv, ax_delta) = plt.subplots(1, 2, figsize=(12, 4))
nearest.plot(x="strike", y="iv", ax=ax_iv, title=f"Implied volatility by strike ({nearest['expiration'].iloc[0].date()})", legend=False)
nearest.plot(x="strike", y="delta", ax=ax_delta, title="Call delta by strike", legend=False)
for ax in (ax_iv, ax_delta):
    ax.axvline(spot, linestyle="--", color="grey")
plt.tight_layout()

## 2. Pick a contract: about 30 days out, about 0.30 delta

A 0.30-delta call roughly 30 days out is a common strike for covered calls. Contracts without a bid or without Greeks are skipped.

In [ ]:
today = pd.Timestamp.today().normalize()
calls["dte"] = (calls["expiration"] - today).dt.days
liquid = calls[(calls["dte"] >= 7) & (calls["bid"] > 0) & calls["delta"].notna()]
if liquid.empty:
    raise RuntimeError("No liquid call contracts with Greeks in this chain; try another underlying.")

target_expiration = liquid.loc[(liquid["dte"] - 30).abs().idxmin(), "expiration"]
candidates = liquid[liquid["expiration"] == target_expiration]
pick = candidates.loc[(candidates["delta"] - 0.30).abs().idxmin()]
OCC = pick["occ_symbol"].strip()

print(f"Selected {OCC}: strike {pick['strike']}, {pick['dte']} days to expiry, delta {pick['delta']:.2f}, IV {pick['iv']:.1%}")
pick[["strike", "bid", "ask", "iv", "delta", "gamma", "theta", "vega"]]

## 3. Daily history of that contract

`client.options_history()` accepts an OCC contract symbol (padded or unpadded) and returns daily bars with OHLC, bid/ask, volume, open interest, implied volatility and Greeks.

In [ ]:
history = client.options_history(OCC, limit=90)
bars = to_dataframe(history, "bars").rename(columns={"implied_volatility": "iv", "greeks.delta": "delta", "greeks.theta": "theta"})
if bars.empty:
    raise RuntimeError(f"No daily bars yet for {OCC}; newly listed contracts have little history. Pick another contract above.")

bars["date"] = pd.to_datetime(bars["date"])
bars = bars.set_index("date").sort_index()
print(f"{len(bars)} daily bars for {OCC}")
bars[["open", "high", "low", "close", "volume", "open_interest", "iv", "delta", "theta"]].tail()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
bars["close"].plot(ax=axes[0], title=f"{OCC} close")
bars["iv"].plot(ax=axes[1], title="Implied volatility")
bars["delta"].plot(ax=axes[2], title="Delta")
plt.tight_layout()

## 4. Strategy analysis: selling the call

Sell one contract (100 shares) at the first day's close and mark it to market at every close after. The second line estimates how much of that came from time decay alone, by adding up each day's theta (an approximation: theta changes with price and volatility).

In [ ]:
entry = bars["close"].iloc[0]
short_call_pnl = (entry - bars["close"]) * 100
theta_collected = (-bars["theta"]).shift(1).fillna(0).cumsum() * 100

analysis = pd.DataFrame({"short call P&L ($)": short_call_pnl, "time decay collected ($)": theta_collected})
analysis.plot(figsize=(10, 4), title=f"Short 1 {OCC} from {bars.index[0].date()}")
print(f"Sold at {entry:.2f}; last close {bars['close'].iloc[-1]:.2f}. "
      f"P&L ${short_call_pnl.iloc[-1]:,.0f}, of which time decay about ${theta_collected.iloc[-1]:,.0f}. "
      f"IV moved {bars['iv'].iloc[0]:.1%} -> {bars['iv'].iloc[-1]:.1%}.")

## 5. Expired contracts (Quant & Community+)

Pass the contract terms instead of an OCC symbol to backtest contracts that have already expired. On Trader Pro this June 2024 contract is outside the 90-day window, so it returns no bars.

In [ ]:
try:
    expired = client.options_history("AAPL", expiration="2024-06-21", contract_type="call", strike=200, limit=250)
    expired_bars = to_dataframe(expired, "bars")
    if expired_bars.empty:
        print("No bars returned: history beyond the trailing 90 days needs Quant & Community or above.")
    else:
        print(f"{expired.get('occ_symbol', '').strip()}: {len(expired_bars)} daily bars")
        display(expired_bars.reindex(columns=["date", "close", "implied_volatility", "greeks.delta", "open_interest"]).tail())
except ApexTickAPIError as err:
    print(f"HTTP {err.status_code}: {err.message}")

## Next steps

- Swap `UNDERLYING`, the target delta or the days to expiry to study other setups.
- Use `contract_type="put"` for cash-secured puts.
- Field reference: [docs.apextick.com](https://docs.apextick.com). Quotes and OHLCV basics: [`01_batched_quotes_and_ohlcv_quickstart.ipynb`](https://colab.research.google.com/github/apextick/apextick-starter-kit/blob/main/notebooks/01_batched_quotes_and_ohlcv_quickstart.ipynb).